# Remote job salaries: a first look

This notebook walks through the **Remote Job Salaries by Role and Region** dataset: what it contains, which remote roles post the highest pay, and how to read the numbers without overreaching.

The data is compiled by [Remote Landers](https://remotelanders.com), a remote job board where every listing states a salary range and comes from the employer's own careers page. Each row is an aggregate (a count of open postings plus pay percentiles), not an individual job.

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Find the CSV wherever Kaggle mounted the dataset
paths = sorted(glob.glob('/kaggle/input/**/job_salary_aggregate_*.csv', recursive=True)) \
        or sorted(glob.glob('job_salary_aggregate_*.csv'))
df = pd.read_csv(paths[-1])
print(paths[-1])
df.head()

## What is in the snapshot

One row per **role × region scope × pay unit**. `annual_usd` rows are yearly pay for full-time roles; `hourly_usd` rows are hourly pay. Percentile columns are empty when a row's pay comes from too few employers to represent the market, so it is worth checking how many rows have them.

In [ ]:
print(f"Rows: {len(df)}")
print(f"Rows with salary percentiles: {df['salary_p50'].notna().sum()}")
print(f"Distinct roles: {df['category'].nunique()}")
pd.crosstab(df['region_scope'], df['salary_unit'], margins=True)

Most rows are **US-only**. No role has 5 or more *worldwide* postings in this snapshot, so `Worldwide` does not appear at all, which says something about how "remote" most remote jobs are.

Note that `listing_count` cannot be summed across roles: a posting tagged with two roles is counted in both rows.

## Highest-paying remote roles (US-only, annual)

Bars show the 10th to 90th percentile of posted pay; the dot is the median. Only rows with published percentiles are included.

In [ ]:
NAVY, LIGHT, YELLOW = '#033F8F', '#99C9FF', '#FFD100'

us = (df[(df['region_scope'] == 'US-only') & (df['salary_unit'] == 'annual_usd')]
      .dropna(subset=['salary_p50'])
      .sort_values('salary_p50'))
top = us.tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
for i, r in enumerate(top.itertuples()):
    color = YELLOW if r.category == 'Software Engineer' else LIGHT
    ax.plot([r.salary_p10, r.salary_p90], [i, i], color=color, lw=9, solid_capstyle='round', zorder=1)
    ax.scatter(r.salary_p50, i, s=90, color=NAVY, edgecolor='white', lw=1.5, zorder=2)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([f"{c}  (n={n})" for c, n in zip(top['category'], top['listing_count'])])
ax.xaxis.set_major_formatter(lambda x, _: f"${x/1000:,.0f}k")
ax.set_xlabel('Posted annual pay, US dollars (10th to 90th percentile, dot = median)')
ax.set_title('Top 15 US-only remote roles by median posted pay')
ax.grid(axis='x', alpha=0.3)
ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.show()

Wide bars matter as much as high medians. Roles like enterprise sales and legal have a very wide 10th–90th spread, so a single "average salary" would hide a lot. Small `n` values (shown next to each role) mean the range can shift noticeably between snapshots.

## Does saying "US-only" change the pay?

Some employers restrict a remote role to US residents; others don't say where applicants can be based (`Unspecified`). For roles with percentiles under both scopes, here are the medians side by side.

In [ ]:
annual = df[(df['salary_unit'] == 'annual_usd') & df['salary_p50'].notna()]
wide = annual.pivot_table(index='category', columns='region_scope', values='salary_p50')
both = wide.dropna(subset=['US-only', 'Unspecified'])[['US-only', 'Unspecified']]
both['difference'] = both['US-only'] - both['Unspecified']
higher = (both['difference'] > 0).sum()
print(f"US-only median is higher in {higher} of {len(both)} roles")
both.sort_values('difference', ascending=False).style.format('${:,.0f}')

In most of these roles the US-only median is higher, usually by a few thousand dollars, with bigger gaps in a few roles such as recruiting and accounting. Many of these rows are small, so treat individual gaps as rough. Also, `Unspecified` is not the same as remote-anywhere: it only means the posting did not say where applicants can be based.

## Hourly roles

Hourly pay is reported separately and is not converted to yearly, because hourly roles are often part-time or contract.

In [ ]:
hourly = df[(df['salary_unit'] == 'hourly_usd') & df['salary_p50'].notna()].sort_values('salary_p50')

fig, ax = plt.subplots(figsize=(9, 0.6 * len(hourly) + 1.5))
for i, r in enumerate(hourly.itertuples()):
    ax.plot([r.salary_p10, r.salary_p90], [i, i], color=LIGHT, lw=9, solid_capstyle='round', zorder=1)
    ax.scatter(r.salary_p50, i, s=90, color=NAVY, edgecolor='white', lw=1.5, zorder=2)
ax.set_yticks(range(len(hourly)))
ax.set_yticklabels([f"{c} ({s}, n={n})" for c, s, n in zip(hourly['category'], hourly['region_scope'], hourly['listing_count'])])
ax.xaxis.set_major_formatter(lambda x, _: f"${x:,.0f}")
ax.set_xlabel('Posted hourly pay, US dollars (10th to 90th percentile, dot = median)')
ax.set_title('Remote roles with hourly pay')
ax.grid(axis='x', alpha=0.3)
ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.show()

## Reading this data responsibly

- Only postings that **state pay in US dollars** are included, so this describes pay-transparent remote postings, not all remote jobs.
- Figures are **posted ranges** (the midpoint of each range), not what people end up earning.
- An empty percentile means the row's pay came from too few employers; `listing_count` is still accurate.
- Roles are tagged automatically from posting text, so a few postings will sit under the wrong role.

## Source and license

Data: [Remote Landers](https://remotelanders.com), 2026-10 snapshot, licensed CC BY 4.0. If you reuse it, please credit "Remote Landers" and link to https://remotelanders.com/data/remote-job-salaries. Suggested citation: Remote Landers, *Remote Job Salaries*, 2026-10 snapshot. https://remotelanders.com/data/remote-job-salaries Questions or corrections: contact@remotelanders.com